In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/Research/FINAL/Code/MMMM

/content/drive/MyDrive/Research/FINAL/Code/MMMM


In [3]:
!pip install -q scipy
!pip install -q h5py
!pip install -q matplotlib
!pip install -q torch torchvision torchaudio
!pip install -q transformers
!pip install -q fuzzy_match
!pip install -q nltk
!pip install -q rouge
!pip install -q diffusers
!pip install -q tensorboard
!pip install -q tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.4 MB/s eta 0:00:00


In [4]:
import sys
import time

sys.path.append("./models")
sys.path.append("./training")
sys.path.append("./testing")
sys.path.append("./utils")
sys.path.append("./ZuCo")

from master_init import *
from DSG import *
from data import *
from dataloader import *
from count_params import *

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.tensorboard import SummaryWriter

from transformers import BartTokenizer

# from argparser import get_config

In [5]:
# @title Hyperparameters

device = "cuda:0" # @param
device_ids = [0] # @param
learning_rate = 5e-3 # @param

optimizer = optim.Adam # @param
criterion = nn.CrossEntropyLoss() # @param

In [6]:
model = INITIALIZE_MODEL(device=device, device_ids=device_ids).to(device)
model = nn.DataParallel(model, device_ids=device_ids)
optimizer = optimizer(model.parameters(), lr=learning_rate)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

In [8]:
dataloaders = INITIALIZE_DATALOADERS(
        keys=["ZuCo-BART"],
        bsz=[64]
    )
ZuCo_dataloader=dataloaders["ZuCo-BART"]
print("[INFO] Intialized dataloaders.")

[INFO] Intialized dataloaders.


In [9]:
def load_img_data(dir="./data/Brain2Image"):
    with open(f"{dir}/imageNet_labeled_eeg.pkl", "rb") as f:
        image_eeg_labels = pickle.load(f)
    with open(f"{dir}/image_net_dict.pkl", "rb") as f:
        img_net_dict = pickle.load(f)
    return {"data" : image_eeg_labels, "targets" : img_net_dict}

In [10]:
brain2imgdata=load_img_data()

In [17]:
for key, val in brain2imgdata['data']['dev'].items():
    print(key)

eeg
labels


In [20]:
Brain2Image_dataloader=ImageNetDataloader(labeled_eeg=brain2imgdata['data']['dev'], image_net_dict=brain2imgdata['targets'])

In [ ]:
args_dict

In [ ]:
model("EEG-IMG-BRAIN2IMAGE", args_dict)